#### Phase 1.1 — Load & Structural Inspection

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

: 

In [ ]:
file_path = "../data/Online Retail.xlsx"

df = pd.read_excel(file_path)

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

#### Phase 1.2 — Data Quality Profiling

In [ ]:
# Missing values
df.isna().sum()

In [ ]:
# Missing value percentages
(df.isna().mean() * 100).round(2)

In [ ]:
# Exact duplicate rows
df.duplicated().sum()
#(df.duplicated().mean()*100).round(2)

In [ ]:
df[["Quantity", "UnitPrice"]].describe()

### Initial Data Quality Findings

- CustomerID is missing in 24.93% of transaction rows, which may limit customer-level analysis.
- Description has a relatively small number of missing values (0.27%).
- The dataset contains 5,268 exact duplicate rows that require validation.
- Quantity contains both positive and negative values, requiring investigation of cancellations and returns.
- UnitPrice contains zero/negative and extreme values that require further inspection before revenue calculations.

#### Phase 1.3 — Investigating Transaction Anomalies

What do negative quantities, cancellation invoices, and unusual prices actually represent?

In [ ]:
quantity_summary = pd.Series({
    "Positive": (df["Quantity"] > 0).sum(),
    "Zero": (df["Quantity"] == 0).sum(),
    "Negative": (df["Quantity"] < 0).sum()
})

quantity_summary

In [ ]:
cancellation_mask = df["InvoiceNo"].astype(str).str.startswith("C")

cancellation_mask.sum()

In [ ]:
df.loc[cancellation_mask, "InvoiceNo"].nunique()

In [ ]:
pd.crosstab(
    df["Quantity"] < 0,
    cancellation_mask,
    rownames=["Negative Quantity"],
    colnames=["Cancellation Invoice"]
)

All invoice lines explicitly marked as cancellations have negative quantities, but not all negative-quantity records are cancellation invoices.

In [ ]:
price_summary = pd.Series({
    "Positive": (df["UnitPrice"] > 0).sum(),
    "Zero": (df["UnitPrice"] == 0).sum(),
    "Negative": (df["UnitPrice"] < 0).sum()
})

price_summary

In [ ]:
df[df["UnitPrice"] < 0]

Records with negative UnitPrice correspond to bad-debt accounting adjustments rather than product sales and are therefore excluded from sales KPIs.

In [ ]:
df[df["Quantity"].abs() == 80995]

#### Phase 1.4 — Investigating 1,336 non-negative and non-C rows 

If some negative quantities are not cancellations, what type of records are they?

In [ ]:
non_cancel_negative = df[
    (df["Quantity"] < 0) &
    (~cancellation_mask)
]

non_cancel_negative.head(20)

In [ ]:
non_cancel_negative["Description"].value_counts(dropna=False).head(20)

In [ ]:
non_cancel_negative["StockCode"].value_counts().head(20)

In [ ]:
zero_price = df[df["UnitPrice"] == 0]

zero_price.head(20)

In [ ]:
zero_price["Description"].value_counts(dropna=False).head(20)

In [ ]:
pd.crosstab(
    zero_price["Quantity"] < 0,
    zero_price["CustomerID"].isna(),
    rownames=["Negative Quantity"],
    colnames=["Missing CustomerID"]
)

In [ ]:
pd.crosstab(
    zero_price["Quantity"] < 0,
    zero_price["CustomerID"].isna(),
    rownames=["Negative Quantity"],
    colnames=["Missing CustomerID"]
)

In [ ]:
missing_description = df[df["Description"].isna()]

missing_description[
    ["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID"]
].head(20)

In [ ]:
pd.Series({
    "Total": len(missing_description),
    "Negative Quantity": (missing_description["Quantity"] < 0).sum(),
    "Zero Price": (missing_description["UnitPrice"] == 0).sum(),
    "Missing CustomerID": missing_description["CustomerID"].isna().sum()
})

Negative quantities without cancellation invoice numbers represent operational or inventory adjustments rather than customer sales transactions.

Further investigation showed that negative quantities were not exclusively
customer cancellations. All 1,336 negative-quantity records without a
"C"-prefixed invoice had zero unit prices and missing customer IDs, with
descriptions such as "damages", "thrown away", and "check". These records
appear to represent internal inventory adjustments and should not be treated
as customer sales or returns.

In [ ]:
df[
    (df["UnitPrice"] == 0) &
    (df["CustomerID"].notna())
][
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "CustomerID"
    ]
]

Zero-priced transactions are retained in the cleaned source data but excluded from paid-sales KPIs and product sales rankings because their commercial meaning cannot be reliably determined.

In [ ]:
duplicate_rows = df[
    df.duplicated(keep=False)
].sort_values(
    ["InvoiceNo", "StockCode", "InvoiceDate"]
)

duplicate_rows.head(20)

In [ ]:
len(duplicate_rows)

The raw dataset contained 5,268 exact duplicate rows. Since all available
transaction attributes were identical, only one copy of each duplicated
record was retained to avoid double-counting sales, quantities, and revenue.

## Phase 2 - Data Cleaning

In [ ]:
df_clean = df.copy()

#### Step 2.1 — Exact duplicates

In [ ]:
rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

rows_after = len(df_clean)

print(f"Rows before: {rows_before:,}")
print(f"Rows after:  {rows_after:,}")
print(f"Removed:     {rows_before - rows_after:,}")

#### Step 2.2 - Transaction Flags

In [ ]:
df_clean["IsCancellation"] = (
    df_clean["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

df_clean["IsZeroPrice"] = df_clean["UnitPrice"] == 0

df_clean["IsNegativePrice"] = df_clean["UnitPrice"] < 0

df_clean["HasCustomerID"] = df_clean["CustomerID"].notna()

In [ ]:
df_clean["IsOperationalAdjustment"] = (
    (df_clean["Quantity"] < 0) &
    (~df_clean["IsCancellation"]) &
    (df_clean["UnitPrice"] == 0) &
    (df_clean["CustomerID"].isna())
)

In [ ]:
df_clean["CustomerID"] = df_clean["CustomerID"].astype("Int64")

In [ ]:
df_clean[
    [
        "IsCancellation",
        "IsZeroPrice",
        "IsNegativePrice",
        "IsOperationalAdjustment"
    ]
].sum()

#### Step 2.3 - Dataset Definition

Which transactions represent actual paid customer purchases?

sales_df

In [ ]:
sales_df = df_clean[
    (~df_clean["IsCancellation"]) &
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

cancellation_df

In [ ]:
cancellations_df = df_clean[
    (df_clean["IsCancellation"]) &
    (df_clean["Quantity"] < 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

customer_df

In [ ]:
customer_df = sales_df[
    sales_df["CustomerID"].notna()
].copy()

In [ ]:
sales_df["Revenue"] = (
    sales_df["Quantity"] * sales_df["UnitPrice"]
)

In [ ]:
cancellations_df["Revenue"] = (
    cancellations_df["Quantity"] * cancellations_df["UnitPrice"]
)

In [ ]:
customer_df = sales_df[
    sales_df["CustomerID"].notna()
].copy()

In [ ]:
print(f"Clean rows:         {len(df_clean):,}")
print(f"Paid sales rows:    {len(sales_df):,}")
print(f"Cancellation rows:  {len(cancellations_df):,}")
print(f"Customer sales rows:{len(customer_df):,}")

In [ ]:
revenue_summary = pd.Series({
    "Gross Revenue": sales_df["Revenue"].sum(),
    "Cancellation Value": -cancellations_df["Revenue"].sum(),
    "Net Revenue": (
        sales_df["Revenue"].sum()
        + cancellations_df["Revenue"].sum()
    )
})

revenue_summary.apply(lambda x: f"£{x:,.2f}")

In [ ]:
pd.Series({
    "Sales Invoices": sales_df["InvoiceNo"].nunique(),
    "Cancellation Invoices": cancellations_df["InvoiceNo"].nunique(),
    "Customers in Sales Data": customer_df["CustomerID"].nunique()
})

In [ ]:
pd.Series({
    "Paid Sales Rows": len(sales_df),
    "Paid Sales Rows with CustomerID": sales_df["CustomerID"].notna().sum(),
    "Paid Sales Rows without CustomerID": sales_df["CustomerID"].isna().sum(),
    "CustomerID Coverage (%)": sales_df["CustomerID"].notna().mean() * 100
}).round(2)

### Data Cleaning Summary

The raw dataset contained 541,909 transaction lines. After removing 5,268 exact duplicate records, 536,641 rows remained.

Transactions were classified based on their business meaning rather than applying blanket filtering rules. Positive paid transactions were separated from cancellation invoices, zero-priced operational records, and accounting adjustments.

The resulting paid-sales dataset contains 524,878 transaction lines across 19,960 invoices. Customer-level analysis is performed on the subset with identifiable customers, containing 392,692 transaction lines and 4,338 unique customers. Customer IDs are available for 74.82% of paid-sales transaction lines.

Cancellation records were retained separately so that their financial impact can be analyzed rather than being silently discarded.

## Phase 3 — Feature Engineering & KPI Definitions

Business question: 
How should raw transaction lines be transformed into meaningful business metrics?

#### Step 3.1 - Time Features

In [ ]:
sales_df["YearMonth"] = (
    sales_df["InvoiceDate"]
    .dt.to_period("M")
)

In [ ]:
cancellations_df["YearMonth"] = (
    cancellations_df["InvoiceDate"]
    .dt.to_period("M")
)

In [ ]:
customer_df["YearMonth"] = (
    customer_df["InvoiceDate"]
    .dt.to_period("M")
)

In [ ]:
sales_df["Weekday"] = sales_df["InvoiceDate"].dt.day_name()
sales_df["Hour"] = sales_df["InvoiceDate"].dt.hour

#### Step 3.2 - Order Value

In [ ]:
orders_df = (
    sales_df
    .groupby(
        ["InvoiceNo"],
        as_index=False
    )
    .agg(
        InvoiceDate=("InvoiceDate", "min"),
        OrderValue=("Revenue", "sum"),
        Items=("Quantity", "sum"),
        ProductLines=("StockCode", "count"),
        UniqueProducts=("StockCode", "nunique")
    )
)

In [ ]:
len(orders_df)

In [ ]:
assert len(orders_df) == sales_df["InvoiceNo"].nunique()

In [ ]:
orders_df.head()

In [ ]:
orders_df["OrderValue"].describe()

#### Step 3.4 - KPI's definition

AOV ( Average Order Value )

In [ ]:
average_order_value = orders_df["OrderValue"].mean()

Repeat Customer Rate

In [ ]:
customer_order_counts = (
    customer_df
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
)

pd.Series({
    "Customers": len(customer_order_counts),
    "One-time Customers": (customer_order_counts == 1).sum(),
    "Repeat Customers": (customer_order_counts > 1).sum(),
    "Repeat Customer Rate (%)": (
        (customer_order_counts > 1).mean() * 100
    )
}).round(2)


A repeat customer is an identified customer with more than one distinct paid-sales invoice during the observation period.

## Phase 4 - Core Business KPIs

In [ ]:
gross_revenue = sales_df["Revenue"].sum()

cancellation_value = -cancellations_df["Revenue"].sum()

net_transaction_revenue = (
    gross_revenue - cancellation_value
)

total_orders = orders_df["InvoiceNo"].nunique()

total_customers = customer_df["CustomerID"].nunique()

average_order_value = orders_df["OrderValue"].mean()

identified_customer_revenue = customer_df["Revenue"].sum()

revenue_per_customer = (
    identified_customer_revenue / total_customers
)

repeat_customer_rate = (
    (customer_order_counts > 1).mean() * 100
)

cancellation_value_rate = (
    cancellation_value / gross_revenue * 100
)

identified_revenue_coverage = (
    identified_customer_revenue
    / gross_revenue
    * 100
)

In [ ]:
kpi_display = pd.Series({
    "Gross Revenue": f"£{gross_revenue:,.2f}",
    "Cancellation Value": f"£{cancellation_value:,.2f}",
    "Net Transaction Revenue": f"£{net_transaction_revenue:,.2f}",
    "Total Orders": f"{total_orders:,}",
    "Identified Customers": f"{total_customers:,}",
    "Average Order Value": f"£{average_order_value:,.2f}",
    "Revenue per Identified Customer": f"£{revenue_per_customer:,.2f}",
    "Repeat Customer Rate": f"{repeat_customer_rate:.2f}%",
    "Cancellation Value / Gross Revenue": f"{cancellation_value_rate:.2f}%",
    "Identified customer revenue coverage": f"{identified_revenue_coverage:.2f}%"
})

kpi_display

2,845 of 4,338 identified customers placed more than one paid order during the observation period, resulting in a repeat customer rate of 65.58%.

#### Phase 4A — Sales Performance Analysis

How did sales performance change over time?

In [ ]:
print("Start date:", sales_df["InvoiceDate"].min())
print("End date:  ", sales_df["InvoiceDate"].max())

In [ ]:
monthly_sales = (
    sales_df
    .groupby("YearMonth", as_index=False)
    .agg(
        GrossRevenue=("Revenue", "sum"),
        Orders=("InvoiceNo", "nunique")
    )
)

monthly_sales["AverageOrderValue"] = (
    monthly_sales["GrossRevenue"] / monthly_sales["Orders"]
)

monthly_sales

#### Step 4A-2 - Cancellation

In [ ]:
monthly_cancellations = (
    cancellations_df
    .groupby("YearMonth", as_index=False)
    .agg(
        CancellationValue=("Revenue", lambda x: -x.sum()),
        CancellationInvoices=("InvoiceNo", "nunique")
    )
)

In [ ]:
monthly_performance = monthly_sales.merge(
    monthly_cancellations,
    on="YearMonth",
    how="left"
)

monthly_performance[
    ["CancellationValue", "CancellationInvoices"]
] = monthly_performance[
    ["CancellationValue", "CancellationInvoices"]
].fillna(0)

In [ ]:
monthly_performance["NetTransactionRevenue"] = (
    monthly_performance["GrossRevenue"]
    - monthly_performance["CancellationValue"]
)

In [ ]:
monthly_performance["MoMGrossRevenueGrowth"] = (
    monthly_performance["GrossRevenue"]
    .pct_change()
    * 100
)

In [ ]:
monthly_performance["CancellationValueRate"] = (
    monthly_performance["CancellationValue"]
    / monthly_performance["GrossRevenue"]
    * 100
)

In [ ]:
monthly_performance.round({
    "GrossRevenue": 2,
    "AverageOrderValue": 2,
    "CancellationValue": 2,
    "NetTransactionRevenue": 2,
    "CancellationValueRate": 2,
    "MoMGrossRevenueGrowth": 2,
    "CancellationValueRate": 2
})

In [ ]:
monthly_performance.loc[
    monthly_performance["GrossRevenue"].idxmax()
]

In [ ]:
monthly_complete = monthly_performance[
    monthly_performance["YearMonth"] != pd.Period("2011-12", freq="M")
].copy()

November 2011 was the strongest complete month in the dataset, generating approximately £1.50M in gross revenue from 2,769 paid orders.

The strong revenue growth from September through November was primarily associated with higher order volume rather than increasing average order value.

#### Step 4A.3 - Visualization

How did monthly revenue evolve over the observation period?

In [ ]:
monthly_performance["Month"] = (
    monthly_performance["YearMonth"].dt.to_timestamp()
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    monthly_performance["Month"],
    monthly_performance["GrossRevenue"],
    marker="o",
    label="Gross Revenue"
)

plt.plot(
    monthly_performance["Month"],
    monthly_performance["NetTransactionRevenue"],
    marker="o",
    label="Net Transaction Revenue"
)

plt.title("Monthly Gross vs Net Transaction Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    monthly_performance["Month"],
    monthly_performance["Orders"],
    marker="o"
)

plt.title("Monthly Paid Orders")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.grid(alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Finding 1 — Peak sales

November 2011 was the strongest complete month, with approximately £1.50M in gross revenue and 2,769 paid orders.

Finding 2 — Growth mechanism

Revenue accelerated substantially from September through November. The increase was accompanied by strong growth in order volume, while average order value remained relatively stable and even declined slightly, indicating that higher order volume was the primary driver.

Finding 3 — Cancellation variability

Cancellation value varied considerably across months. January 2011 recorded cancellation value equivalent to roughly 19% of gross revenue, compared with approximately 3.2% in November.

Note: December 2011 is a partial month containing transactions only through December 9 and is therefore excluded from full-month performance comparisons.

#### Phase 4A.4 — Revenue by Country

Which geographic markets contribute the most to sales revenue?

In [ ]:
country_performance = (
    sales_df
    .groupby("Country", as_index=False)
    .agg(
        GrossRevenue=("Revenue", "sum"),
        Orders=("InvoiceNo", "nunique")
    )
    .sort_values("GrossRevenue", ascending=False)
)

In [ ]:
country_performance["RevenueSharePct"] = (
    country_performance["GrossRevenue"]
    / country_performance["GrossRevenue"].sum()
    * 100
)

In [ ]:
country_performance["AverageOrderValue"] = (
    country_performance["GrossRevenue"]
    / country_performance["Orders"]
)

In [ ]:
country_performance[
    [
        "Country",
        "GrossRevenue",
        "Orders",
        "AverageOrderValue",
        "RevenueSharePct"
    ]
].head(10).round(2)

In [ ]:
uk_performance = country_performance[
    country_performance["Country"] == "United Kingdom"
]

uk_performance

In [ ]:
pd.Series({
    "Number of Countries": sales_df["Country"].nunique(),
    "UK Revenue Share (%)": uk_performance["RevenueSharePct"].iloc[0],
    "UK Gross Revenue": uk_performance["GrossRevenue"].iloc[0],
    "UK Orders": uk_performance["Orders"].iloc[0]
}).round(2)

In [ ]:
top_non_uk = (
    country_performance[
        country_performance["Country"] != "United Kingdom"
    ]
    .head(10)
)

top_non_uk[
    ["Country", "GrossRevenue", "RevenueSharePct", "Orders"]
]

In [ ]:
top_non_uk_plot = (
    country_performance[
        country_performance["Country"] != "United Kingdom"
    ]
    .head(10)
    .sort_values("GrossRevenue")
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_non_uk_plot["Country"],
    top_non_uk_plot["GrossRevenue"]
)

plt.title("Top Non-UK Markets by Gross Revenue")
plt.xlabel("Gross Revenue (£)")
plt.ylabel("Country")

plt.tight_layout()
plt.show()

In [ ]:
country_performance[
    country_performance["Orders"] >= 20
].sort_values(
    "AverageOrderValue",
    ascending=False
)[
    [
        "Country",
        "Orders",
        "AverageOrderValue",
        "GrossRevenue"
    ]
].head(10).round(2)

Geographic Sales Performance: The United Kingdom dominates the business, accounting for 84.59% of gross revenue. Among international markets, purchasing patterns differ substantially. The Netherlands and Australia generate relatively high revenue despite modest order counts, with average order values of approximately £3,037 and £2,429 respectively, compared with about £500 in the UK. Germany and France, by contrast, generate revenue primarily through higher order volumes with average order values closer to the UK level. These differences suggest that international markets should be evaluated not only by total revenue but also by order size and purchasing behavior.

#### Phase 4B - Product Performance

In [ ]:
product_performance = (
    sales_df
    .groupby("StockCode", as_index=False)
    .agg(
        QuantitySold=("Quantity", "sum"),
        GrossRevenue=("Revenue", "sum"),
        Orders=("InvoiceNo", "nunique")
    )
)

In [ ]:
product_names = (
    sales_df
    .groupby("StockCode")["Description"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .reset_index()
)

In [ ]:
product_performance = product_performance.merge(
    product_names,
    on="StockCode",
    how="left"
)

In [ ]:
len(product_performance)

Top Products by Revenue

In [ ]:
top_products_revenue = (
    product_performance
    .sort_values("GrossRevenue", ascending=False)
    .head(10)
)

top_products_revenue[
    [
        "StockCode",
        "Description",
        "GrossRevenue",
        "QuantitySold",
        "Orders"
    ]
].round(2)

Top Products by Quantity

In [ ]:
top_products_quantity = (
    product_performance
    .sort_values("QuantitySold", ascending=False)
    .head(10)
)

top_products_quantity[
    [
        "StockCode",
        "Description",
        "QuantitySold",
        "GrossRevenue",
        "Orders"
    ]
].round(2)

Gross product rankings can be distorted by large cancelled transactions, therefore product performance should also be evaluated after considering cancellations.

In [ ]:
non_product_codes = [
    "DOT",
    "POST",
    "M"
]

sales_df[
    sales_df["StockCode"].isin(non_product_codes)
][
    ["StockCode", "Description"]
].drop_duplicates()

Product Sales Dataset

In [ ]:
product_sales_df = sales_df[
    ~sales_df["StockCode"].isin(non_product_codes)
].copy()

In [ ]:
product_sales_df["StockCode"].nunique()

In [ ]:
product_performance = (
    product_sales_df
    .groupby("StockCode", as_index=False)
    .agg(
        QuantitySold=("Quantity", "sum"),
        GrossRevenue=("Revenue", "sum"),
        Orders=("InvoiceNo", "nunique")
    )
)

In [ ]:
product_names = (
    product_sales_df
    .groupby("StockCode")["Description"]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else x.iloc[0]
    )
    .reset_index()
)

product_performance = product_performance.merge(
    product_names,
    on="StockCode",
    how="left"
)

In [ ]:
sales_df[
    sales_df["Description"].str.contains(
        "POST|Manual",
        case=False,
        na=False
    )
]["Description"].value_counts()

In [ ]:
top_products_revenue = (
    product_performance
    .sort_values("GrossRevenue", ascending=False)
    .head(10)
)

top_products_revenue[
    [
        "StockCode",
        "Description",
        "GrossRevenue",
        "QuantitySold",
        "Orders"
    ]
]

Should cancelled products be removed from product performance? YES

### Product Anakysis Types
1. Gross Product Performance:
    Which products generated the highest recorded sales?
2. Net Product Performance:
    Which products contributed the most after cancellations?

In [ ]:
product_performance["RevenuePerOrder"] = (
    product_performance["GrossRevenue"]
    / product_performance["Orders"]
)

In [ ]:
product_performance.sort_values(
    "RevenuePerOrder",
    ascending=False
).head(10)

In [ ]:
product_cancellations = (
    cancellations_df
    .groupby("StockCode", as_index=False)
    .agg(
        CancellationValue=("Revenue", lambda x: -x.sum())
    )
)

In [ ]:
product_net = product_performance.merge(
    product_cancellations,
    on="StockCode",
    how="left"
)

In [ ]:
product_net["CancellationValue"] = (
    product_net["CancellationValue"]
    .fillna(0)
)

In [ ]:
product_net["NetRevenue"] = (
    product_net["GrossRevenue"]
    - product_net["CancellationValue"]
)

In [ ]:
product_net.sort_values(
    "NetRevenue",
    ascending=False
).head(10)

In [ ]:
top10_share = (
    product_net
    .sort_values("NetRevenue", ascending=False)
    .head(10)["NetRevenue"]
    .sum()
    /
    product_net["NetRevenue"].sum()
    * 100
)

top10_share